In [ ]:
suppressPackageStartupMessages({
    library(monocle3)
    library(MASS)
    library(reshape2)
    library(VGAM)
    library(dplyr)
    library(ggplot2)
    library(grid)
    library(gridExtra)
    library(Matrix)
    library(IRdisplay)
    library(tibble)
    library(ggridges)
    library(piano)
    library(pheatmap)
    library(stringr)
    library(scales)
    library(foreach)
    library(ggpubr)
    library(DelayedArray)
})

In [ ]:
parametricDispersionFit <- function( disp_table, verbose = FALSE, initial_coefs=c(1e-6, 1) )
{
    coefs <- initial_coefs
    iter <- 0
    while(TRUE) {
        residuals <- disp_table$disp / ( coefs[1] + coefs[2] / disp_table$mu )
        good <- disp_table[which(disp_table$disp > 0 & (residuals < 10000) ),]
        if(verbose)
            fit <- glm( disp ~ I(1/mu), data=good,
                        family=Gamma(link="identity"), start=coefs )
        else
            suppressWarnings(fit <- glm( disp ~ I(1/mu), data=good,
                                         family=Gamma(link="identity"), start=coefs ))
        
        oldcoefs <- coefs
        coefs <- coefficients(fit)
        if (coefs[1] < initial_coefs[1]){
            coefs[1] <- initial_coefs[1]
        }
        if (coefs[2] < 0){
            stop( "Parametric dispersion fit failed. Try a local fit and/or a pooled estimation. (See '?estimateDispersions')" )
        }
        if( sum( log( coefs / oldcoefs )^2 ) < coefs[1] )
            break
        iter <- iter + 1

        if( iter > 10 ) {
            warning( "Dispersion fit did not converge." )
            break
        }
    }
    
    if( !all( coefs > 0 ) ){
        stop( "Parametric dispersion fit failed. Try a local fit and/or a pooled estimation. (See '?estimateDispersions')" )
    }
    rm(disp_table)
    
    list(fit, coefs)
}

disp_calc_helper_NB <- function(cds, expressionFamily, min_cells_detected){
    
    rounded <- round(exprs(cds))
    nzGenes <- Matrix::rowSums(rounded > 0.1) 
    nzGenes <- names(nzGenes[nzGenes > min_cells_detected])
    
    x <- DelayedArray(t(t(rounded[nzGenes,]) / size_factors(cds[nzGenes,])))
    
    xim <- mean(1/ size_factors(cds[nzGenes,]))
    
    
    f_expression_mean <- as(DelayedMatrixStats::rowMeans2(x), "sparseVector")
    f_expression_var <- DelayedMatrixStats::rowVars(x)
    
    disp_guess_meth_moments <- f_expression_var - xim * f_expression_mean
    
    disp_guess_meth_moments <- disp_guess_meth_moments / (f_expression_mean^2) #fix the calculation of k
    
    
    res <- data.frame(mu=as.vector(f_expression_mean), disp=as.vector(disp_guess_meth_moments))
    res[res$mu == 0]$mu = NA
    res[res$mu == 0]$disp = NA
    res$disp[res$disp < 0] <- 0
    
    res <- cbind(gene_id=row.names(fData(cds[nzGenes,])), res)
    res
}

estimateDispersionsForCellDataSet <- function(cds, min_cells_detected, removeOutliers, verbose = FALSE)
{
    mu <- NA
    
    progress_opts <- options()$dplyr.show_progress
    options(dplyr.show_progress = T)
    
    cds_pdata <- dplyr::group_by_(dplyr::select_(rownames_to_column(as.data.frame(pData(cds))), "rowname"))
    disp_table <- as.data.frame(disp_calc_helper_NB(cds, cds@expressionFamily, min_cells_detected))

    if(!is.list(disp_table))
        stop("Parametric dispersion fitting failed, please set a different lowerDetectionLimit")

    disp_table <- subset(disp_table, is.na(mu) == FALSE)
    res <- parametricDispersionFit(disp_table, verbose)
    fit <- res[[1]]
    coefs <- res[[2]]

    if (removeOutliers){
        CD <- cooks.distance(fit)
        cooksCutoff <- 4/nrow(disp_table)
        message (paste("Removing", length(CD[CD > cooksCutoff]), "outliers"))
        outliers <- union(names(CD[CD > cooksCutoff]), setdiff(row.names(disp_table), names(CD)))
        res <- parametricDispersionFit(disp_table[row.names(disp_table) %in% outliers == FALSE,], verbose)
        fit <- res[[1]]
        coefs <- res[[2]]
    }
    names( coefs ) <- c( "asymptDisp", "extraPois" )
    ans <- function( q )
        coefs[1] + coefs[2] / q
    attr( ans, "coefficients" ) <- coefs
    
    res <- list(disp_table = disp_table, disp_func = ans)
    return(res)
}
    
dispersionTable <- function (disp_results) 
{
    if (is.null(disp_results)) {
        warning("Warning: estimateDispersions only works, and is only needed, when you're using a CellDataSet with a negbinomial or negbinomial.size expression family")
        stop("Error: no dispersion model found. Please call estimateDispersions() before calling this function")
    }
    disp_df <- data.frame(gene_id = disp_results$disp_table$gene_id, 
                          mean_expression = disp_results$disp_table$mu, 
                          dispersion_fit = disp_results$disp_func(disp_results$disp_table$mu), 
                          dispersion_empirical = disp_results$disp_table$disp)
    return(disp_df)
}

In [ ]:
cds_VCS = readRDS("../data/cds_VCS_with_cell_type_labels_E15-15.5_1e-4_subclustered_score_all_genes_umap2_added.RDS")

umap_coords <- reducedDims(cds_VCS)[['UMAP']]
cds_VCS$UMAP1 <- umap_coords[,1]
cds_VCS$UMAP2 <- umap_coords[,2]
cds_VCS$UMAP3 <- umap_coords[,3]

In [ ]:
traj_configs <- list(
    "Chondrocytes" = list(
        traj_name = "Mesoderm",
        cell_filter = "Early",
        umap_filter = expression(UMAP2 < 3.8),
        res_k = 700,
        learn_graph = list(ncenter = 300, maxiter = 20, minimal_branch_len = 10),
        root_filter = expression(UMAP3 < 8.4 & UMAP2 < 4)
    ),
    "Osteoblasts" = list(
        traj_name = "Mesoderm",
        cell_filter = "osteo",
        umap_filter = expression(!(UMAP2 > 5 & UMAP3 < 9.5)),
        res_k = 1000,
        learn_graph = list(ncenter = 300, maxiter = 20, minimal_branch_len = 10),
        root_filter = expression(UMAP2 < 3.5)
    ),
    "Glutamatergic" = list(
        traj_name = "CNS_neurons",
        cell_filter = "Glutamatergic",
        umap_filter = expression(UMAP3 < 0.5),
        res_res = 2e-4,
        cluster_remove = c(32, 39),
        res_res_2 = 2e-4, 
        learn_graph = list(ncenter = 600, maxiter = 10, minimal_branch_len = 5),
        root_filter = expression(UMAP2 < -6.5)
    ),
    "GABAergic" = list(
        traj_name = "CNS_neurons",
        cell_filter = "GABAergic n",
        umap_filter = expression(UMAP3 < 0.5),
        res_res = 2e-4,
        cluster_remove = c(22),
        res_res_2 = 2e-4,
        learn_graph = list(ncenter = 600, maxiter = 10, minimal_branch_len = 5),
        root_filter = expression(UMAP2 < -6.5)
    ),
    "Definitive_erythroid" = list(
        traj_name = "Definitive_erythroid",
        cell_filter = "",
        umap_filter = NULL,
        res_res = 1.5e-4,
        learn_graph = list(ncenter = 500, maxiter = 10, minimal_branch_len = 5),
        root_filter = expression(UMAP2 > 0)
    ),
    "Upper_layer_neurons" = list(
        traj_name = "Intermediate_progenitors",
        cell_filter = "Intermediate|Upper",
        umap_filter = expression(UMAP2 > 0.3),
        res_res = 5e-4,
        cluster_remove = c(6, 21, 12, 13, 9),
        res_res_2 = 1e-4,
        learn_graph = list(ncenter = 300, maxiter = 20, minimal_branch_len = 10),
        root_filter = expression(UMAP2 > 2.8)
    ),
    "Astrocytes" = list(
        traj_name = "Brain_and_spinal_cord",
        cell_filter = "Astrocytes",
        umap_filter = expression(UMAP2 < 3.8),
        res_res = 4e-4,
        learn_graph = list(ncenter = 300, maxiter = 20, minimal_branch_len = 10),
        root_filter = expression(UMAP1 < (-0.45 * UMAP3 + 8.7))
    )
    
    "Hepatocytes" = list(
        traj_name = "Hepatocytes",
        cell_filter = "", # No specific cell subtype filter
        umap_filter = expression(UMAP2 > 6.3),
        res_res = 5e-4,
        cluster_remove = c(24, 25, 27, 28, 30),
        res_res_2 = 3e-4,
        learn_graph = list(ncenter = 450, maxiter = 10, minimal_branch_len = 5),
        root_filter = expression(UMAP3 > -8)
    )
)

In [ ]:
pseudotime_list <- list()

for (target in names(traj_configs)) {
    config <- traj_configs[[target]]

    if (config$cell_filter != "") {
      cds_sub <- cds_VCS[, cds_VCS$trajectory_refined == config$traj_name & 
                            grepl(config$cell_filter, cds_VCS$nn_majority_cell_type_umap)]
    } else {
      cds_sub <- cds_VCS[, cds_VCS$trajectory_refined == config$traj_name]
    }

    if (!is.null(config$umap_filter)) {
      cds_sub <- cds_sub[, eval(config$umap_filter, envir = as.data.frame(colData(cds_sub)))]
    }

    if (!is.null(config$res_k)) {
      cds_sub <- cluster_cells(cds_sub, cluster_method = "louvain", k = config$res_k)
    } else {
      cds_sub <- cluster_cells(cds_sub, resolution = config$res_res)
    }

    cds_sub$Cluster_sub_2 <- clusters(cds_sub)

    if (!is.null(config$cluster_remove)) {
      cds_sub <- cds_sub[, !cds_sub$Cluster_sub_2 %in% config$cluster_remove]
      cds_sub <- cluster_cells(cds_sub, resolution = config$res_res_2)
    }

    lg_ctrl <- config$learn_graph
    cds_sub <- learn_graph(cds_sub, 
                         learn_graph_control = list(ncenter = lg_ctrl$ncenter, 
                                                    maxiter = lg_ctrl$maxiter, 
                                                    minimal_branch_len = lg_ctrl$minimal_branch_len))

    root_filter_eval <- eval(config$root_filter, envir = as.data.frame(reducedDims(cds_sub)[["UMAP"]]))
    root_cells <- rownames(as.data.frame(reducedDims(cds_sub)[["UMAP"]][root_filter_eval, ]))

    if (target == "Astrocytes") {
      metadata_sub <- as.data.frame(colData(cds_sub))
      root_cells <- rownames(metadata_sub[metadata_sub$UMAP1 < (-0.45 * metadata_sub$UMAP3 + 8.7), ])
    }

    cat("Number of root cells:", length(root_cells), "\n")
    cds_sub <- order_cells(cds_sub, root_cells = root_cells)
    cds_sub$Pseudotime <- pseudotime(cds_sub)
    cds_sub$Cluster_sub_2 <- clusters(cds_sub)

    subcluster_df <- as.data.frame(colData(cds_sub)) %>% select(Cell, Cluster_sub_2, Pseudotime)
    pseudotime_list[[target]] <- subcluster_df
}

In [ ]:
## Fibroblasts
## There were different subtypes in our data (e.g. Tnc+)
## but we focused on subtypes with Fap expression

cds_fibro <- cds_VCS[, cds_VCS$trajectory_refined == "Mesoderm" & 
                       grepl("Fibro", cds_VCS$nn_majority_cell_type_umap)]
cds_fibro <- cds_fibro[, !(cds_fibro$UMAP2 > 5 & cds_fibro$UMAP3 < 9.5)]



disp_table <- estimateDispersionsForCellDataSet(cds_fibro, 500, FALSE) %>% dispersionTable()
disp_table <- disp_table %>% 
                filter(dispersion_empirical > 0) %>%
                mutate(excess_disp = (dispersion_empirical - dispersion_fit) / dispersion_fit) %>%
                arrange(-excess_disp)


top_subset_genes <- as.character(head(disp_table, 500)$gene_id)
custom_genes <- c("Col1a1", "Col1a2", "Col5a1", "Cd34", "Pdgfra", "Fap", "Postn", 
                  "Col12a1", "Thbs1", "Tnc", "Egfl6", "Twist2", "Pax3", "Sim1", 
                  "Foxf1", "Hoxb6", "Hand1", "Hand2", "Prrx1")

custom_gene_ids <- as.data.frame(rowData(cds_fibro)) %>% 
                      filter(gene_short_name %in% custom_genes) %>% pull(gene_ID) %>% as.character()
final_genes <- unique(c(custom_gene_ids, top_subset_genes))


cds_fibro <- preprocess_cds(cds_fibro, num_dim = 30, use_genes = final_genes)
cds_fibro <- align_cds(cds_fibro, alignment_k = 20, residual_model_formula_str = "~ log(RNA_UMIs)")
cds_fibro <- reduce_dimension(cds_fibro, reduction_method = "UMAP", max_components = 2, 
                              cores = 6, umap.n_neighbors = 200, umap.min_dist = 0.4, umap.metric = 'cosine')
cds_fibro <- cluster_cells(cds_fibro, resolution = 3e-5)
cds_fibro$Cluster_sub_2 <- clusters(cds_fibro)


fibroblasts_anno <- data.frame(
  Cluster_sub_2 = as.factor(c(2, 4, 5, 6, 13, 10, 7, 11)),
  subtype = c("Col11a1+", "Postn+", "Postn+", "Cd34+", "Cd34+", "Fap+", "Fap+", "Tnc+")
)

cds_fibro <- cds_fibro[, cds_fibro$Cluster_sub_2 %in% fibroblasts_anno$Cluster_sub_2]
cds_fibro$subtype <- as.data.frame(colData(cds_fibro)) %>% 
                        left_join(fibroblasts_anno, by = "Cluster_sub_2") %>% pull(subtype)


cds_fibro <- cluster_cells(cds_fibro, resolution = 2e-4)
cds_fibro <- cds_fibro[, !clusters(cds_fibro) %in% c(18, 7, 1, 25, 21)]
cds_fibro <- cds_fibro[, cds_fibro$subtype %in% c("Postn+", "Cd34+", "Fap+")]

cds_fibro <- cluster_cells(cds_fibro, resolution = 3e-4)
cds_fibro <- learn_graph(cds_fibro, learn_graph_control = list(ncenter=300, maxiter=10, minimal_branch_len=10))


root_filter <- reducedDims(cds_fibro)[["UMAP"]][,2] > 5.8
root_cells <- rownames(as.data.frame(reducedDims(cds_fibro)[["UMAP"]][root_filter, ]))

cds_fibro <- order_cells(cds_fibro, root_cells = root_cells)
cds_fibro$Pseudotime <- pseudotime(cds_fibro)
cds_fibro$Cluster_sub_2 <- clusters(cds_fibro)

pseudotime_list[["Fibroblasts_Fap+"]] <- as.data.frame(colData(cds_fibro)) %>% select(Cell, Cluster_sub_2, Pseudotime)

saveRDS(cds_fibro, "cds_fibroblasts_fap.RDS")